In [ ]:
import os
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import cv2
import numpy as np
import qrcode
from PIL import Image

In [ ]:
DATASET_DIR = "dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")

QR_SIZE = 21
PATCH_SIZE = 10

DATASET_SIZE = 40_000
IMAGE_SIZE = QR_SIZE * PATCH_SIZE

START_DATETIME = datetime(2023, 1, 1, 0, 0, 0)
END_DATETIME = datetime(2026, 12, 31, 23, 59, 59)

In [ ]:
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(LABELS_DIR, exist_ok=True)

In [ ]:
def get_random_datetime(start: datetime, end: datetime) -> datetime:
    total_seconds = int((end - start).total_seconds())
    return start + timedelta(seconds=random.randint(0, total_seconds))


def build_qr_code(payload: str) -> Tuple[List[List[bool]], np.ndarray]:
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=1,
        border=0,
    )
    qr.add_data(payload)
    qr.make(fit=False)

    matrix = qr.get_matrix()
    base = Image.new("1", (QR_SIZE, QR_SIZE), 1)
    pixel_access = base.load()
    
    for y in range(QR_SIZE):
        for x in range(QR_SIZE):
            pixel_access[x, y] = 0 if matrix[y][x] else 1

    image = base.resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST).convert("L")
    return matrix, np.array(image)

In [ ]:
def chance(probability: float) -> bool:
    return random.random() < probability


def resize(image: np.ndarray) -> np.ndarray:
    return cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)


def apply_padding(image: np.ndarray) -> np.ndarray:
    height, width = image.shape

    min_ratio = -0.02
    max_ratio = 0.12

    padding_top = int(round(random.uniform(min_ratio, max_ratio) * height))
    padding_bottom = int(round(random.uniform(min_ratio, max_ratio) * height))
    padding_left = int(round(random.uniform(min_ratio, max_ratio) * width))
    padding_right = int(round(random.uniform(min_ratio, max_ratio) * width))

    crop_top = max(0, -padding_top)
    crop_bottom = max(0, -padding_bottom)
    crop_left = max(0, -padding_left)
    crop_right = max(0, -padding_right)

    if crop_top or crop_bottom or crop_left or crop_right:
        image = image[
            crop_top : height - crop_bottom if crop_bottom else height,
            crop_left : width - crop_right if crop_right else width,
        ]

    border_top = max(0, padding_top)
    border_bottom = max(0, padding_bottom)
    border_left = max(0, padding_left)
    border_right = max(0, padding_right)

    if border_top or border_bottom or border_left or border_right:
        border_value = random.randint(180, 255)
        image = cv2.copyMakeBorder(
            image,
            border_top,
            border_bottom,
            border_left,
            border_right,
            borderType=cv2.BORDER_CONSTANT,
            value=border_value,
        )

    return image


def apply_rotation(image: np.ndarray) -> np.ndarray:
    if not chance(0.50):
        return image

    height, width = image.shape
    angle = random.uniform(-8.0, 8.0)
    scale = random.uniform(0.96, 1.04)

    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, scale)

    return cv2.warpAffine(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=random.randint(180, 255),
    )


def apply_perspective(image: np.ndarray) -> np.ndarray:
    if not chance(0.70):
        return image

    height, width = image.shape
    margin = random.uniform(0.01, 0.06) * min(width, height)

    src = np.float32([
        [0, 0],
        [width, 0],
        [width, height],
        [0, height],
    ])

    dst = np.float32([
        [random.uniform(0, margin), random.uniform(0, margin)],
        [width - random.uniform(0, margin), random.uniform(0, margin)],
        [width - random.uniform(0, margin), height - random.uniform(0, margin)],
        [random.uniform(0, margin), height - random.uniform(0, margin)],
    ])

    matrix = cv2.getPerspectiveTransform(src, dst)

    return cv2.warpPerspective(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=random.randint(180, 255),
    )


def apply_curl(image: np.ndarray) -> np.ndarray:
    if not chance(0.35):
        return image

    height, width = image.shape
    x, y = np.meshgrid(np.arange(width), np.arange(height))

    amplitude_x = random.uniform(0.5, 3.5)
    amplitude_y = random.uniform(0.0, 1.5)
    frequency_x = random.uniform(0.5, 1.6)
    frequency_y = random.uniform(0.5, 1.4)
    phase_x = random.uniform(0, 2 * np.pi)
    phase_y = random.uniform(0, 2 * np.pi)

    map_x = x + amplitude_x * np.sin(2 * np.pi * frequency_x * y / height + phase_x)
    map_y = y + amplitude_y * np.sin(2 * np.pi * frequency_y * x / width + phase_y)

    return cv2.remap(
        image,
        map_x.astype(np.float32),
        map_y.astype(np.float32),
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=random.randint(180, 255),
    )


def apply_anisotropic_scale(image: np.ndarray) -> np.ndarray:
    if not chance(0.60):
        return image

    height, width = image.shape
    scale_x = random.uniform(0.94, 1.06)
    scale_y = random.uniform(0.94, 1.06)

    scaled = cv2.resize(
        image,
        None,
        fx=scale_x,
        fy=scale_y,
        interpolation=cv2.INTER_LINEAR,
    )

    scaled_height, scaled_width = scaled.shape

    if scaled_height >= height:
        y0 = (scaled_height - height) // 2
        scaled = scaled[y0:y0 + height, :]
    else:
        padding_top = (height - scaled_height) // 2
        padding_bottom = height - scaled_height - padding_top
        scaled = cv2.copyMakeBorder(
            scaled,
            padding_top,
            padding_bottom,
            0,
            0,
            cv2.BORDER_CONSTANT,
            value=random.randint(180, 255),
        )

    scaled_height, scaled_width = scaled.shape

    if scaled_width >= width:
        x0 = (scaled_width - width) // 2
        scaled = scaled[:, x0:x0 + width]
    else:
        padding_left = (width - scaled_width) // 2
        padding_right = width - scaled_width - padding_left
        scaled = cv2.copyMakeBorder(
            scaled,
            0,
            0,
            padding_left,
            padding_right,
            cv2.BORDER_CONSTANT,
            value=random.randint(180, 255),
        )

    return scaled


def apply_morphology(image: np.ndarray) -> np.ndarray:
    if not chance(0.40):
        return image

    kernel_size = random.choice([2, 2, 3])
    kernel = np.ones((kernel_size, kernel_size), np.uint8)

    morphology_operation = random.choice([
        cv2.MORPH_ERODE,
        cv2.MORPH_DILATE,
        cv2.MORPH_OPEN,
        cv2.MORPH_CLOSE,
    ])

    return cv2.morphologyEx(image, morphology_operation, kernel, iterations=1)


def apply_gray(image: np.ndarray) -> np.ndarray:
    gray_level = random.randint(150, 235)
    texture_sigma = random.uniform(2.0, 6.0)

    texture = np.random.normal(0, texture_sigma, image.shape)
    paper_background = np.full(image.shape, gray_level, dtype=np.float32) + texture

    height, width = image.shape

    coarse_texture = np.random.normal(
        0,
        random.uniform(2.0, 8.0),
        (8, 8),
    ).astype(np.float32)

    coarse_texture = cv2.resize(
        coarse_texture,
        (width, height),
        interpolation=cv2.INTER_CUBIC,
    )

    paper_background += coarse_texture

    output = np.where(
        image > 128,
        paper_background,
        image.astype(np.float32),
    )

    return np.clip(output, 0, 255).astype(np.uint8)


def apply_black_lift(image: np.ndarray) -> np.ndarray:
    if chance(0.40):
        black_level = random.randint(0, 15)
    else:
        black_level = random.randint(15, 60)

    dark_noise = np.random.normal(
        0,
        random.uniform(2.0, 5.0),
        image.shape,
    )

    lifted_black = black_level + dark_noise

    output = np.where(
        image < 128,
        lifted_black,
        image.astype(np.float32),
    )

    return np.clip(output, 0, 255).astype(np.uint8)


def apply_lighting(image: np.ndarray) -> np.ndarray:
    if not chance(0.85):
        return image

    height, width = image.shape

    y_coordinates, x_coordinates = np.meshgrid(
        np.arange(height),
        np.arange(width),
        indexing="ij",
    )

    center_x = random.uniform(-0.4 * width, 1.4 * width)
    center_y = random.uniform(-0.4 * height, 1.4 * height)

    distance = np.sqrt(
        (x_coordinates - center_x) ** 2
        + (y_coordinates - center_y) ** 2
    )

    distance /= max(distance.max(), 1)

    strength = random.uniform(0.05, 0.28)
    mask = 1.0 - strength * distance

    if chance(0.25):
        band_center = random.uniform(-0.25 * width, 1.25 * width)
        band_width = random.uniform(0.06 * width, 0.20 * width)

        band = np.exp(
            -((x_coordinates - band_center) ** 2) / (2 * band_width ** 2)
        )

        mask += random.uniform(0.06, 0.22) * band

    global_gain = random.uniform(0.75, 1.18)

    output = image.astype(np.float32) * mask * global_gain
    output += random.uniform(-8, 10)

    return np.clip(output, 0, 255).astype(np.uint8)


def apply_blur(image: np.ndarray) -> np.ndarray:
    if not chance(0.18):
        return image

    blur_type = random.choice(["gaussian", "gaussian", "motion"])

    if blur_type == "gaussian":
        kernel_size = random.choice([(3, 3), (3, 3), (5, 5)])
        blur_sigma = random.uniform(0.45, 1.6)

        return cv2.GaussianBlur(
            image,
            kernel_size,
            sigmaX=blur_sigma,
        )

    motion_kernel_size = random.choice([3, 5])
    motion_kernel = np.zeros(
        (motion_kernel_size, motion_kernel_size),
        dtype=np.float32,
    )

    if chance(0.5):
        motion_kernel[motion_kernel_size // 2, :] = 1.0
    else:
        motion_kernel[:, motion_kernel_size // 2] = 1.0

    motion_kernel /= motion_kernel.sum()

    return cv2.filter2D(image, -1, motion_kernel)


def apply_contrast(image: np.ndarray) -> np.ndarray:
    contrast_alpha = random.uniform(0.80, 1.25)
    brightness_beta = random.uniform(-18, 18)

    return cv2.convertScaleAbs(
        image,
        alpha=contrast_alpha,
        beta=brightness_beta,
    )


def apply_noise(image: np.ndarray) -> np.ndarray:
    if not chance(0.45):
        return image

    output = image.astype(np.float32)

    if chance(0.80):
        gaussian_noise_sigma = random.uniform(1.5, 7.0)

        output += np.random.normal(
            0,
            gaussian_noise_sigma,
            image.shape,
        )

    if chance(0.25):
        salt_and_pepper_amount = random.uniform(0.001, 0.01)
        salt_and_pepper_mask = np.random.random(image.shape)

        output[salt_and_pepper_mask < salt_and_pepper_amount / 2] = random.randint(0, 30)
        output[salt_and_pepper_mask > 1 - salt_and_pepper_amount / 2] = random.randint(220, 255)

    return np.clip(output, 0, 255).astype(np.uint8)


def apply_paper_defects(image: np.ndarray) -> np.ndarray:
    if not chance(0.25):
        return image

    output = image.astype(np.float32)
    height, width = image.shape

    for _ in range(random.randint(1, 3)):
        start_x = random.randint(-width // 8, width)
        start_y = random.randint(-height // 8, height)

        end_x = int(
            np.clip(
                start_x + random.randint(-width // 3, width // 3),
                0,
                width - 1,
            )
        )

        end_y = int(
            np.clip(
                start_y + random.randint(-height // 3, height // 3),
                0,
                height - 1,
            )
        )

        defect_color = random.randint(120, 235)
        defect_thickness = random.choice([1, 1, 2])

        cv2.line(
            output,
            (start_x, start_y),
            (end_x, end_y),
            defect_color,
            defect_thickness,
        )

    return np.clip(output, 0, 255).astype(np.uint8)


def apply_jpeg(image: np.ndarray) -> np.ndarray:
    if not chance(0.30):
        return image

    jpeg_quality = random.randint(40, 92)

    encoding_success, encoded_image = cv2.imencode(
        ".jpg",
        image,
        [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality],
    )

    if not encoding_success:
        return image

    decoded_image = cv2.imdecode(encoded_image, cv2.IMREAD_GRAYSCALE)

    return decoded_image if decoded_image is not None else image


def augment_image(image: np.ndarray) -> np.ndarray:
    image = image.astype(np.uint8)
    image = resize(image)

    image = apply_padding(image)
    image = apply_rotation(image)
    image = apply_perspective(image)
    image = apply_curl(image)
    image = apply_anisotropic_scale(image)
    image = resize(image)

    image = apply_morphology(image)
    image = apply_gray(image)
    image = apply_black_lift(image)
    image = apply_lighting(image)
    image = apply_blur(image)
    image = apply_contrast(image)
    image = apply_noise(image)
    image = apply_paper_defects(image)
    image = apply_jpeg(image)

    return np.clip(image, 0, 255).astype(np.uint8)

In [ ]:
def write_label(path: str, matrix: List[List[bool]]) -> None:
    with open(path, "w", encoding="utf-8") as file:
        for row in matrix:
            file.write(" ".join("1" if v else "0" for v in row) + "\n")

In [ ]:
used_names = set()
for i in range(DATASET_SIZE):
    while True:
        datetime = get_random_datetime(START_DATETIME, END_DATETIME)
        basename = datetime.strftime("%d%m%Y%H%M%S")
        if basename not in used_names:
            used_names.add(basename)
            break

    payload = datetime.strftime("%d-%m-%Y %H:%M:%S")

    matrix, qr_code = build_qr_code(payload)
    augmented_qr_code = augment_image(qr_code)

    image_path = os.path.join(IMAGES_DIR, f"{basename}.png")
    label_path = os.path.join(LABELS_DIR, f"{basename}.txt")
    
    cv2.imwrite(image_path, augmented_qr_code)
    write_label(label_path, matrix)

    if (i+1) % 500 == 0:
        print(f"{i+1}/{DATASET_SIZE} images generated.")